In [24]:
import nilearn 
import os.path as op
from nilearn import image
import os
import numpy as np
import pandas as pd

bids_folder = '/Volumes/mrenkeED/data/ds-stressrisk'


In [43]:
from stress_risk.utils import Subject
from nilearn import surface

sub=1
subject = f'{sub:02d}'
ses = 1
session = ses
derivatives = op.join(bids_folder, 'derivatives')
base_dir = 'glm_nilearn.residuals'
base_dir = op.join(derivatives, base_dir, f'sub-{subject}', f'ses-{session}', 'func')
os.makedirs(base_dir, exist_ok=True)

sub = Subject(sub, bids_folder=bids_folder)

ims = sub.get_preprocessed_bold_surf(session=ses, space='fsaverage5', hemis=['L'])
data = [surface.load_surf_data(im) for im in ims]



file exists:True


In [36]:
value=False
onsets = sub.get_fmri_events(session=session, runs = range(1, 7), value=value)
tr = 2.3
n = 135
frametimes = np.linspace(tr/2., (n - .5)*tr, n)
onsets['onset'] = ((onsets['onset']+tr/2.) // 2.3) * 2.3
onsets.drop(columns=['n2', 'trial_nr'], inplace=True)

In [45]:
from nilearn.glm.first_level import make_first_level_design_matrix

dm = [make_first_level_design_matrix(frametimes, onsets.loc[run], hrf_model='fir', oversampling=100.,
                                         drift_order=0,
                                         drift_model=None).drop('constant', axis=1) for run in runs]

dm = pd.concat(dm, keys=range(1, 7), names=['run']).fillna(0)
dm.columns = [c.replace('_delay_0', '') for c in dm.columns]
dm /= dm.max()
#print(dm)
dm[dm < 1.0] = 0.0
print(dm.shape)

X = [dm.loc[run].values for run in range(1, 7)]
print(len(X))
print(np.shape(data))


(810, 142)
6
(6, 10242, 135)


In [51]:
ims = sub.get_preprocessed_bold_surf(session=ses, space='fsaverage5', hemis=['L'])
data = [surface.load_surf_data(im) for im in ims]

data = np.concatenate(data, axis=1)
print(np.shape(data))


file exists:True
(10242, 810)


In [52]:
from nilearn.glm.first_level import FirstLevelModel

glm = FirstLevelModel()
glm = glm.fit(data.T, design_matrices=dm)

TypeError: Data given cannot be loaded because it is not compatible with nibabel format:
1090.8186

In [ ]:
# chatGPT starter code
from nilearn.input_data import NiftiMasker
from nilearn.glm.first_level import FirstLevelModel
from nilearn.glm.first_level.design_matrix import make_design_matrix

fmri_img = 'path/to/your/fmri_data.nii.gz'

# Load the mask image
mask_img = 'path/to/your/mask.nii.gz'

# Mask the fMRI data
masker = NiftiMasker(mask_img=mask_img, standardize=True)
masked_data = masker.fit_transform(fmri_img)

# Define the timing and duration of task events
# For example, using make_design_matrix function
design_matrix = make_design_matrix(frame_times, events)

# Initialize the FirstLevelModel object with the design matrix
glm = FirstLevelModel()
glm = glm.fit(masked_data, design_matrices=design_matrix)

# Get residuals
residuals = glm.residuals[0]  # Assuming there is only one session

In [ ]:

import pandas as pd

events = pd.DataFrame({
    'onset': [10, 30, 50],       # Onset times of events (in seconds)
    'duration': [5, 5, 5],        # Durations of events (in seconds)
    'trial_type': ['task1', 'task2', 'task1']  # Condition labels
})


file exists:True


/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nilearn/glm/__init__.py:55: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  warn('The nilearn.glm module is experimental. '
/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
ims

['/Volumes/mrenke/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-1_space-fsaverage_desc-preproc_bold.nii.gz',
 '/Volumes/mrenke/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-2_space-fsaverage_desc-preproc_bold.nii.gz',
 '/Volumes/mrenke/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-3_space-fsaverage_desc-preproc_bold.nii.gz',
 '/Volumes/mrenke/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-4_space-fsaverage_desc-preproc_bold.nii.gz',
 '/Volumes/mrenke/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-5_space-fsaverage_desc-preproc_bold.nii.gz',
 '/Volumes/mrenke/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-6_space-fsaverage_desc-preproc_bold.nii.gz']

In [13]:
subject = '01'
session = 1
space = 'fsaverage5'
runs = range(1, 7)
hemis = ['L', 'R']
images = [op.join(bids_folder, 'derivatives', 'fmriprep', f'sub-{subject}',
f'ses-{session}', 'func', f'sub-{subject}_ses-{session}_task-risk_run-{run}_space-{space}_hemi-{hemi}_bold.func.gii') for run in runs for hemi in hemis]

images

['/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-1_space-fsaverage5_hemi-L_bold.func.gii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-1_space-fsaverage5_hemi-R_bold.func.gii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-2_space-fsaverage5_hemi-L_bold.func.gii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-2_space-fsaverage5_hemi-R_bold.func.gii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-3_space-fsaverage5_hemi-L_bold.func.gii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-3_space-fsaverage5_hemi-R_bold.func.gii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-4_space-fsave

In [14]:
for im in images:
    print(im)
    print(op.exists(im))

/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-1_space-fsaverage5_hemi-L_bold.func.gii
True
/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-1_space-fsaverage5_hemi-R_bold.func.gii
True
/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-2_space-fsaverage5_hemi-L_bold.func.gii
True
/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-2_space-fsaverage5_hemi-R_bold.func.gii
True
/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-3_space-fsaverage5_hemi-L_bold.func.gii
True
/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-3_space-fsaverage5_hemi-R_bold.func.gii
True
/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-4_space-f

In [ ]:
'/Volumes/mrenke/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-6_space-fsaverage5_hemi-R_bold.func.gii'
'/Volumes/mrenkeED/data/ds-stressrisk/derivatives/fmriprep/sub-01/ses-1/func/sub-01_ses-1_task-risk_run-1_space-fsnative_hemi-R_bold.func.gii'